# Final PCA: per-sample-set covariate PCs

Ancestry filtering happens upstream of this notebook now: round 1 (`01_premade_label_filter.ipynb`) selects a base cohort from AoU's premade continental label, round 2 (`04_round2_1000g_filter.ipynb`) fits a genuine 1000G-referenced Mahalanobis ellipsoid and writes each final `SAMPLE_SET`'s keep-list. This notebook's only job is producing the covariate PCs `02_residualize_phenotypes.ipynb` reads.

**Why refit rather than reuse round 2's PCs:** round 2's PCA is fit on the 1000G reference space (for classification, at `BASE_GROUP` granularity); PCs computed on that broader/differently-fit space don't correctly reflect a tighter final `SAMPLE_SET`'s own internal structure. So this notebook restricts `03_genome_wide_qc_thinning_merge.ipynb`'s shared `BASE_GROUP` panel to a `SAMPLE_SET`'s own final members (`--keep`, cheap -- no re-QC/re-pruning) and refits `--pca approx` down to `N_PCS=10` *within exactly that cohort*.

Further thins the ~1M-variant GRM panel to ~100K variants first (GRM wants that density, PCA doesn't need nearly as much) -- same calibrate-then-apply `--thin` pattern `01_king_po_exclusion.ipynb` uses for its own kinship-specific thinning. That thinning step runs once per `BASE_GROUP` (cached, shared across every `SAMPLE_SET` under it), then the per-`SAMPLE_SET` `--keep` + refit runs on top of it.</cell id="fpca-intro">

## Compute resource

Same panel `03_grm_shards/02_grm_shard_timing.ipynb`/`03a_grm_shard_run.ipynb` size for initially, but this notebook runs on the ~100K-variant thinned subset, not the full ~1M-variant GRM panel -- 8-16 vCPU is plenty, no need to size up to the GRM notebooks' level.

## Setup

plink2: manual install, same pattern as everywhere else in this repo.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  # URL is dated; if it 404s, get current link from https://www.cog-genomics.org/plink/2.0/
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

export PATH="$BIN_DIR:$PATH"
plink2 --version
nproc
free -h

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

N_THREADS = os.cpu_count()

## Sample set configuration

Every final `SAMPLE_SET` maps onto a `BASE_GROUP` and a `prob_tag` -- must match `04_round2_1000g_filter.ipynb`'s own `SAMPLE_SETS` dict (`ellipsoid_threshold` there -> `f"p{threshold*100:g}"`, or `"unfiltered"` for `eur_premade_label`), since that's the notebook that actually decided membership and named the keep-list file this one reads.</cell id="fpca-config-md">

In [ ]:
SAMPLE_SETS = {
    "eur":               {"base_group": "eur", "prob_tag": "p99.9999"},
    "eur_stringent":     {"base_group": "eur", "prob_tag": "p99"},
    "eur_loose":         {"base_group": "eur", "prob_tag": "p99.999999"},
    "eur_premade_label": {"base_group": "eur", "prob_tag": "unfiltered"},
    "afr":               {"base_group": "afr", "prob_tag": "p99.9"},
}

N_PCS = 10              # final covariate PC count
PCA_N_SNPS_TARGET = 100_000   # GRM wants ~1M variants; PCA converges with far fewer

CDR_VERSION = "v9"
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{CDR_VERSION}/01_ancestry_filtering"

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_grm")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

## Inputs

Run once per `SAMPLE_SET` (all 5). Copies `03_genome_wide_qc_thinning_merge.ipynb`'s shared `BASE_GROUP` panel locally (same convention as everywhere else in this pipeline -- plink2 reading a multi-GB panel repeatedly over the gcsfuse-mounted bucket is much slower than local disk) and locates `04_round2_1000g_filter.ipynb`'s keep-list for this `SAMPLE_SET`.

In [ ]:
SAMPLE_SET = "eur"   # <-- change this and rerun for each of the 5 sample sets
_cfg = SAMPLE_SETS[SAMPLE_SET]
BASE_GROUP = _cfg["base_group"]

PANEL_DIR = f"{ANCESTRY_BUCKET_DIR}/genome_wide_panel_{BASE_GROUP}"
MERGED_NAME = f"genome_wide_thinned_{CDR_VERSION}_{BASE_GROUP}"   # pgen form, from 03_genome_wide_qc_thinning_merge.ipynb
MERGED_PREFIX = os.path.join(LOCAL_WORK_DIR, MERGED_NAME)

for ext in ("pgen", "pvar", "psam"):
    bucket_path = f"{PANEL_DIR}/{MERGED_NAME}.{ext}"
    local_path = f"{MERGED_PREFIX}.{ext}"
    assert os.path.isfile(bucket_path), (
        f"missing merged panel: {bucket_path!r} -- run 03_genome_wide_qc_thinning_merge.ipynb's "
        f"merge section for BASE_GROUP={BASE_GROUP!r} first"
    )
    if not os.path.isfile(local_path):
        import shutil
        shutil.copy(bucket_path, local_path)

PCA_BED_PREFIX = os.path.join(LOCAL_WORK_DIR, f"{MERGED_NAME}_pca_thinned")   # shared across every SAMPLE_SET under this BASE_GROUP, cached

FINAL_PCA_BUCKET_DIR = f"{PANEL_DIR}/final_pca"
SAMPLE_SET_KEEP_PATH = os.path.join(FINAL_PCA_BUCKET_DIR, f"final_keep_ids_{SAMPLE_SET}_{_cfg['prob_tag']}.txt")
assert os.path.isfile(SAMPLE_SET_KEEP_PATH), (
    f"missing keep-list: {SAMPLE_SET_KEEP_PATH!r} -- run 04_round2_1000g_filter.ipynb "
    f"for BASE_GROUP={BASE_GROUP!r} first"
)

SAMPLE_SET_OUT_DIR = os.path.join(FINAL_PCA_BUCKET_DIR, SAMPLE_SET)
os.makedirs(SAMPLE_SET_OUT_DIR, exist_ok=True)
FINAL_PCA_PREFIX = os.path.join(LOCAL_WORK_DIR, f"final_pca_{CDR_VERSION}_{SAMPLE_SET}")

print(MERGED_PREFIX)
print(SAMPLE_SET_KEEP_PATH)
print(SAMPLE_SET_OUT_DIR)

## Further thin for PCA

Same calibrate-then-apply `--thin` pattern as `01_king_po_exclusion.ipynb`'s own kinship-specific thinning of this same panel (50K there, for a different downstream need). Cached and shared across every `SAMPLE_SET` under this `BASE_GROUP` -- skipped on the 2nd+ `SAMPLE_SET` run for the same base group.

In [ ]:
%%bash -s "$MERGED_PREFIX" "$PCA_BED_PREFIX" "$PCA_N_SNPS_TARGET" "$N_THREADS"
set -e
MERGED_PREFIX=$1
PCA_BED_PREFIX=$2
N_TARGET=$3
THREADS=$4

if [ -s "${PCA_BED_PREFIX}.pgen" ]; then
  echo "already thinned, skipping"
else
  N_CURRENT=$(($(wc -l < "${MERGED_PREFIX}.pvar") - 1))
  THIN_P=$(python3 -c "print(min(1.0, ${N_TARGET} / ${N_CURRENT}))")
  echo "current SNPs: $N_CURRENT, target: $N_TARGET, thin_p: $THIN_P"

  plink2 \
    --pfile "$MERGED_PREFIX" \
    --thin "$THIN_P" \
    --threads "$THREADS" \
    --make-pgen \
    --out "$PCA_BED_PREFIX"
fi

echo "Thinned SNP count:"
awk 'END{print NR-1}' "${PCA_BED_PREFIX}.pvar"

In [ ]:
%%bash -s "$PCA_BED_PREFIX" "$SAMPLE_SET_KEEP_PATH" "$FINAL_PCA_PREFIX" "$N_THREADS" "$N_PCS"
set -e
PCA_BED_PREFIX=$1
KEEP_PATH=$2
FINAL_PCA_PREFIX=$3
THREADS=$4
NPCS=$5

plink2 \
  --pfile "$PCA_BED_PREFIX" \
  --keep "$KEEP_PATH" \
  --nonfounders \
  --freq counts \
  --pca approx "$NPCS" \
  --threads "$THREADS" \
  --out "$FINAL_PCA_PREFIX"

ls -lh "${FINAL_PCA_PREFIX}".*

### Scree plot

Quick sanity check -- % variance explained per PC, refit within this exact
`SAMPLE_SET`'s own members.

In [ ]:
import matplotlib.pyplot as plt

eigenval = pd.read_csv(f"{FINAL_PCA_PREFIX}.eigenval", header=None, names=["eigenvalue"])
eigenval["pc"] = range(1, len(eigenval) + 1)
eigenval["pct_variance"] = eigenval["eigenvalue"] / eigenval["eigenvalue"].sum() * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(eigenval["pc"], eigenval["pct_variance"], color="royalblue")
ax.set_xlabel("PC")
ax.set_ylabel("% variance explained")
ax.set_title(f"Final PCA [{SAMPLE_SET}] scree plot")
ax.set_xticks(eigenval["pc"])
plt.tight_layout()
plot_path = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_scree_{SAMPLE_SET}.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")
print(eigenval[["pc", "eigenvalue", "pct_variance"]].to_string(index=False))

## Write PC covariates for 02_residualize_phenotypes.ipynb

Same `IID PC1 ... PC10` format `02_residualize_phenotypes.ipynb`'s `PC_PATH` /
`pull_covariates()` expects.

In [ ]:
direct = pd.read_csv(f"{FINAL_PCA_PREFIX}.eigenvec", sep=r"\s+")

_id_col = "#IID" if "#IID" in direct.columns else "IID"
pc_cols = [c for c in direct.columns if c.startswith("PC")]
assert len(pc_cols) == N_PCS, f"expected {N_PCS} PC columns, found {len(pc_cols)}: {pc_cols}"

covariate_table = direct[[_id_col] + pc_cols].rename(columns={_id_col: "IID"})
covariate_table["IID"] = covariate_table["IID"].astype(str)

PC_COVARIATE_PATH = os.path.join(SAMPLE_SET_OUT_DIR, f"final_pca_pc_covariates_{SAMPLE_SET}.txt")
covariate_table.to_csv(PC_COVARIATE_PATH, sep="\t", index=False)
print(f"Wrote {len(covariate_table)} samples' PC1-PC{N_PCS} -> {PC_COVARIATE_PATH}")

## Next steps

`02_residualize_phenotypes.ipynb` already points its `KEEP_LIST_PATH`/`PC_PATH` at `04_round2_1000g_filter.ipynb`'s/this notebook's output paths for whichever `SAMPLE_SET` is selected -- no manual copying needed.